In [ ]:
%pip install "psycopg[binary]" pgvector numpy


In [1]:
import psycopg
from pgvector.psycopg import register_vector

# Connect to your Postgres instance
conn = psycopg.connect("postgresql://shilpisharma@localhost:5432/postgres")
with conn.cursor() as cur:
    # 1. Enable the pgvector extension inside Postgres
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    conn.commit()
    
    # 2. Register pgvector with psycopg so it understands python lists/numpy arrays as vectors
    register_vector(conn)

print("Extension enabled and registered successfully!")

Extension enabled and registered successfully!


In [3]:
with conn.cursor() as cur:
    # Create a table for documents with a 3-dimensional vector column
    cur.execute("""
        CREATE TABLE IF NOT EXISTS documents (
            id serial PRIMARY KEY,
            content text,
            embedding vector(3)
        );
    """)
    conn.commit()

In [4]:
import numpy as np

# Sample data: Text content mapped to dummy 3D vector embeddings
data = [
    ("How to tune indexes in Postgres", np.array([1.2, 0.3, 0.8])),
    ("Recipes for authentic Italian pasta", np.array([0.1, 0.9, 0.2])),
    ("Optimizing SQL database performance", np.array([1.1, 0.2, 0.9]))
]

with conn.cursor() as cur:
    for content, embedding in data:
        cur.execute(
            "INSERT INTO documents (content, embedding) VALUES (%s, %s);",
            (content, embedding)
        )
    conn.commit()
print("Data inserted successfully!")

Data inserted successfully!


In [5]:
# The 'search query' converted into the same 3D embedding space
query_embedding = np.array([1.1, 0.2, 0.8])

with conn.cursor() as cur:
    # We order by cosine distance (<=>) and limit to the top 2 closest results
    cur.execute("""
        SELECT id, content, embedding <=> %s AS distance 
        FROM documents 
        ORDER BY distance ASC 
        LIMIT 2;
    """, (query_embedding,))
    
    results = cur.fetchall()
    
    for row in results:
        print(f"ID: {row[0]} | Score: {row[2]:.4f} | Content: {row[1]}")

# Close connection when finished
conn.close()

ID: 3 | Score: 0.0016 | Content: Optimizing SQL database performance
ID: 1 | Score: 0.0026 | Content: How to tune indexes in Postgres
